# Aula prática: Python + SQL com o BancoVDE 2026

**Objetivo:** abrir um Excel da raiz do Google Drive, conhecer os dados com Python/pandas e investigar perguntas com SQL. Execute os blocos na ordem (`Shift + Enter`). Tempo sugerido: **60 a 90 minutos**.

**Roteiro:** 1. Importar → 2. Inspecionar → 3. Preparar → 4. Consultar com SQL → 5. Visualizar → 6. Exercícios.

**Pré-requisito:** `BancoVDE 2026.xlsx` na pasta **Meu Drive** (raiz), não dentro de outra pasta. Ao montar o Drive, o Colab solicitará autorização. Não é preciso instalar `pandasql`: usaremos `sqlite3`, já incluído no Python, e `pandas.read_sql_query` para mostrar os resultados em tabelas.

> Cada linha e as colunas `total`, `total_peso` e `abrangencia` precisam ser interpretadas com o dicionário oficial do arquivo. Até lá, as somas abaixo são explorações de **valores registrados**, sem afirmar que correspondem a vítimas únicas ou a taxas populacionais.

## 1. Conectar o Google Drive e encontrar o arquivo

O caminho `/content/drive/MyDrive/` corresponde à raiz de **Meu Drive** no Colab. Se o nome ou a localização forem diferentes, ajuste apenas `nome_arquivo` ou `pasta_drive`.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
pasta_drive = Path('/content/drive/MyDrive')
nome_arquivo = 'BancoVDE 2026.xlsx'
caminho_arquivo = pasta_drive / nome_arquivo

if not caminho_arquivo.is_file():
    parecidos = [p.name for p in pasta_drive.glob('*BancoVDE*') if p.is_file()]
    raise FileNotFoundError(
        f'Arquivo não encontrado na raiz: {caminho_arquivo}. '
        f'Arquivos parecidos nessa pasta: {parecidos}'
    )
print('Arquivo encontrado:', caminho_arquivo)

## 2. Ler a planilha com pandas

`pandas` organiza a planilha em um **DataFrame** (tabela de linhas e colunas). `sheet_name=0` lê a primeira aba; confira os nomes antes de avançar e altere `aba` se os dados estiverem em outra aba.

In [ ]:
import pandas as pd

excel = pd.ExcelFile(caminho_arquivo)
print('Abas disponíveis:', excel.sheet_names)
aba = 0  # Também pode ser um nome, por exemplo: 'Dados'
df_bruto = pd.read_excel(excel, sheet_name=aba)
print('Aba lida:', excel.sheet_names[aba] if isinstance(aba, int) else aba)
print('Dimensões (linhas, colunas):', df_bruto.shape)
display(df_bruto.head(5))

### Conferir estrutura e tipos

`head()` mostra as primeiras linhas; `shape` indica tamanho; `dtypes` mostra o tipo inferido. Aqui retiramos espaços dos nomes das colunas e conferimos se o cabeçalho informado está presente. A conferência falha com uma mensagem clara se a aba escolhida não for a correta.

In [ ]:
colunas_esperadas = [
    'uf', 'municipio', 'evento', 'data_referencia', 'agente', 'arma',
    'faixa_etaria', 'feminino', 'masculino', 'nao_informado',
    'total_vitima', 'total', 'total_peso', 'abrangencia'
]

df = df_bruto.copy()
df.columns = df.columns.astype(str).str.strip().str.lower()
if df.columns.duplicated().any():
    raise ValueError('Há colunas repetidas após retirar espaços e padronizar nomes.')
faltantes = sorted(set(colunas_esperadas) - set(df.columns))
if faltantes:
    raise ValueError(f'Colunas esperadas ausentes: {faltantes}. Confira a aba e o cabeçalho.')

print('Colunas:', df.columns.tolist())
print('Tipos originais:')
display(df.dtypes.rename('tipo').to_frame())

## 3. Primeiros comandos de Python e pandas

Uma variável guarda um valor; uma lista guarda vários; um laço `for` percorre itens; `if` testa uma condição. Para selecionar uma coluna, use `df['coluna']`; para selecionar linhas e colunas, use `df.loc[filtro, colunas]`.

In [ ]:
quantidade_linhas = len(df)
campos_de_interesse = ['uf', 'municipio', 'evento']

for coluna in campos_de_interesse:
    print(f'{coluna}: {df[coluna].nunique(dropna=True)} valores distintos')

if quantidade_linhas == 0:
    print('A aba selecionada não contém linhas de dados.')
else:
    print(f'A planilha contém {quantidade_linhas:,} linhas.')

display(df.loc[:, ['uf', 'municipio', 'evento', 'total_vitima']].head(5))

### Diagnóstico de qualidade, sem alterar os dados

`isna()` encontra campos vazios. `duplicated()` aponta linhas inteiramente iguais, mas não prova erro: registros iguais podem existir legitimamente. Também inspecionamos exemplos das categorias antes de definir filtros.

In [ ]:
diagnostico = pd.DataFrame({
    'vazios': df.isna().sum(),
    'percentual_vazio': (100 * df.isna().mean()).round(1),
    'valores_distintos': df.nunique(dropna=True),
}).sort_values('percentual_vazio', ascending=False)
display(diagnostico)
print('Linhas inteiramente iguais:', int(df.duplicated().sum()))
for coluna in ['uf', 'evento', 'abrangencia']:
    print(f'{coluna} — exemplos:', df[coluna].dropna().astype(str).unique()[:10].tolist())

## 4. Preparar tipos para as consultas

Trabalharemos em `df`, preservando `df_bruto`. Datas inválidas viram `NaT` e números inválidos viram `NaN`; vamos contar as conversões malsucedidas. A conversão numérica abaixo aceita tanto números do Excel quanto textos no formato brasileiro (por exemplo, `1.234,5`). A coluna `total_peso` é convertida, mas não será somada sem conhecer sua definição.

In [ ]:
def numero_do_excel(valor):
    if pd.isna(valor):
        return float('nan')
    if isinstance(valor, str):
        valor = valor.strip().replace(' ', '').replace('\u00a0', '')
        if ',' in valor:
            valor = valor.replace('.', '').replace(',', '.')
    return pd.to_numeric(valor, errors='coerce')

colunas_numericas = [
    'feminino', 'masculino', 'nao_informado', 'total_vitima', 'total', 'total_peso'
]
for coluna in colunas_numericas:
    antes = df[coluna].notna().sum()
    df[coluna] = df[coluna].map(numero_do_excel)
    print(f'{coluna}: {antes - df[coluna].notna().sum()} valor(es) não convertido(s)')

datas_originais = df['data_referencia'].copy()
df['data_referencia'] = pd.to_datetime(
    datas_originais, format='mixed', dayfirst=True, errors='coerce'
)
print('Datas não convertidas (entre as preenchidas):',
      int((datas_originais.notna() & df['data_referencia'].isna()).sum()))
df['uf'] = df['uf'].astype('string').str.strip().str.upper()
display(df[['data_referencia', 'uf', 'total_vitima']].head())

### Checagem exploratória dos totais

Compare `total_vitima` com `feminino + masculino + nao_informado` **apenas nas linhas em que os quatro campos existem**. Uma diferença pode indicar problema de preenchimento, arredondamento ou definições distintas; investigue antes de corrigir valores. Observe separadamente valores negativos, caso não façam sentido para o dicionário de dados.

In [ ]:
componentes = ['feminino', 'masculino', 'nao_informado']
completas = df[componentes + ['total_vitima']].notna().all(axis=1)
diferenca = df.loc[completas, 'total_vitima'] - df.loc[completas, componentes].sum(axis=1)
divergentes = diferenca.abs() > 1e-9
print('Linhas com todos os quatro campos preenchidos:', int(completas.sum()))
print('Linhas com totais divergentes:', int(divergentes.sum()))
display(df.loc[diferenca.index[divergentes],
               ['uf', 'municipio', 'evento'] + componentes + ['total_vitima']].head(10))
print('Valores negativos em total_vitima:', int(df['total_vitima'].lt(0).sum()))

## 5. Tornar a tabela consultável com SQL

`sqlite3.connect(':memory:')` cria um banco temporário. `to_sql` copia o DataFrame para a tabela `vde`; `pd.read_sql_query` executa uma consulta e devolve outro DataFrame. Converter a data em texto `AAAA-MM-DD` permite ordenar e agrupar por mês no SQLite.

In [ ]:
import sqlite3

df_sql = df.copy()
df_sql['data_referencia'] = df_sql['data_referencia'].dt.strftime('%Y-%m-%d')
conexao = sqlite3.connect(':memory:')
df_sql.to_sql('vde', conexao, if_exists='replace', index=False)

def consultar(sql, params=None):
    return pd.read_sql_query(sql, conexao, params=params)

display(consultar('SELECT COUNT(*) AS quantidade_linhas FROM vde'))

## 6. SQL básico: selecionar, filtrar e ordenar

**Anatomia:** `SELECT` escolhe colunas, `FROM` indica a tabela, `WHERE` filtra linhas, `ORDER BY` ordena e `LIMIT` limita o resultado. `SELECT *` é útil para conhecer uma tabela, mas escolher colunas costuma deixar a pergunta mais clara.

In [ ]:
display(consultar("""
SELECT uf, municipio, evento, data_referencia, total_vitima
FROM vde
LIMIT 10
"""))

In [ ]:
display(consultar("""
SELECT uf, municipio, evento, total_vitima
FROM vde
WHERE total_vitima > 0 AND uf IS NOT NULL
ORDER BY total_vitima DESC
LIMIT 10
"""))

### `DISTINCT`, `WHERE` e parâmetros

`DISTINCT` remove valores repetidos do resultado. No filtro, `?` recebe o valor guardado na variável Python sem montar SQL por concatenação. Escolhemos uma UF presente no arquivo para que o exemplo seja executável mesmo se não houver `SP`.

In [ ]:
display(consultar('SELECT DISTINCT uf FROM vde WHERE uf IS NOT NULL ORDER BY uf'))

ufs_disponiveis = df['uf'].dropna()
uf_exemplo = ufs_disponiveis.iloc[0] if not ufs_disponiveis.empty else 'SP'
print('UF escolhida para o exemplo:', uf_exemplo)
display(consultar("""
SELECT municipio, evento, data_referencia, total_vitima
FROM vde
WHERE uf = ? AND total_vitima > ?
ORDER BY data_referencia DESC
LIMIT 10
""", params=(uf_exemplo, 0)))

## 7. Agrupar e agregar

`COUNT(*)` conta **linhas**; `COUNT(total_vitima)` conta linhas com valor informado; `SUM(total_vitima)` soma o indicador. Eles respondem a perguntas diferentes. `GROUP BY` forma grupos e `HAVING` filtra grupos depois da agregação.

In [ ]:
display(consultar("""
SELECT uf,
       COUNT(*) AS registros,
       COUNT(total_vitima) AS registros_com_total_vitima,
       SUM(total_vitima) AS soma_total_vitima
FROM vde
WHERE uf IS NOT NULL
GROUP BY uf
HAVING COUNT(*) >= 1
ORDER BY soma_total_vitima DESC
LIMIT 15
"""))

### Quais eventos aparecem com maior frequência?

Aqui ordenamos pela **quantidade de registros**. Essa pergunta evita presumir que `total_vitima` e `total` tenham a mesma definição. Depois, examine a soma do indicador em uma coluna separada.

In [ ]:
resumo_eventos = consultar("""
SELECT evento,
       COUNT(*) AS registros,
       SUM(total_vitima) AS soma_total_vitima,
       ROUND(AVG(total_vitima), 2) AS media_por_registro
FROM vde
WHERE evento IS NOT NULL
GROUP BY evento
ORDER BY registros DESC
LIMIT 15
""")
display(resumo_eventos)

### Comparar evento e UF

Agrupar por duas colunas responde **em quais UFs cada evento aparece nos registros**. A soma só é interpretável depois de confirmar granularidade, abrangência e eventuais duplicações.

In [ ]:
display(consultar("""
SELECT uf, evento,
       COUNT(*) AS registros,
       SUM(total_vitima) AS soma_total_vitima
FROM vde
WHERE uf IS NOT NULL AND evento IS NOT NULL
GROUP BY uf, evento
ORDER BY registros DESC
LIMIT 20
"""))

## 8. Datas e tendências

`substr(data_referencia, 1, 7)` extrai `AAAA-MM`. Antes de comparar meses, confira se o arquivo cobre todos os períodos e se os critérios de registro permaneceram iguais. Ausência de uma linha no mês não significa necessariamente zero.

In [ ]:
periodos = consultar("""
SELECT substr(data_referencia, 1, 7) AS mes,
       COUNT(*) AS registros,
       SUM(total_vitima) AS soma_total_vitima
FROM vde
WHERE data_referencia IS NOT NULL
GROUP BY substr(data_referencia, 1, 7)
ORDER BY mes
""")
display(periodos)
if not periodos.empty:
    print('Primeiro e último mês:', periodos['mes'].iloc[0], '→', periodos['mes'].iloc[-1])

## 9. `JOIN`: combinar duas tabelas

Vamos criar uma pequena tabela didática com a região de cada UF. `LEFT JOIN` preserva todas as linhas de `vde`, mesmo quando a UF está vazia ou não aparece no cadastro. O grupo `Sem região` merece inspeção.

In [ ]:
regioes_dict = {
    'AC': 'Norte', 'AM': 'Norte', 'AP': 'Norte', 'PA': 'Norte',
    'RO': 'Norte', 'RR': 'Norte', 'TO': 'Norte',
    'AL': 'Nordeste', 'BA': 'Nordeste', 'CE': 'Nordeste',
    'MA': 'Nordeste', 'PB': 'Nordeste', 'PE': 'Nordeste',
    'PI': 'Nordeste', 'RN': 'Nordeste', 'SE': 'Nordeste',
    'DF': 'Centro-Oeste', 'GO': 'Centro-Oeste',
    'MS': 'Centro-Oeste', 'MT': 'Centro-Oeste',
    'ES': 'Sudeste', 'MG': 'Sudeste', 'RJ': 'Sudeste', 'SP': 'Sudeste',
    'PR': 'Sul', 'RS': 'Sul', 'SC': 'Sul',
}
df_regioes = pd.DataFrame(regioes_dict.items(), columns=['uf', 'regiao'])
df_regioes.to_sql('regioes', conexao, if_exists='replace', index=False)

display(consultar("""
SELECT COALESCE(r.regiao, 'Sem região') AS regiao,
       COUNT(*) AS registros,
       SUM(v.total_vitima) AS soma_total_vitima
FROM vde AS v
LEFT JOIN regioes AS r ON v.uf = r.uf
GROUP BY COALESCE(r.regiao, 'Sem região')
ORDER BY registros DESC
"""))

## 10. Visualização rápida com Python

O gráfico de barras mostra **registros por evento** (não vítimas). O gráfico temporal mostra registros por mês. Como o período pode ser incompleto, use o gráfico para formular perguntas e verificar padrões, não para concluir tendência causal.

In [ ]:
import matplotlib.pyplot as plt

if resumo_eventos.empty:
    print('Sem eventos preenchidos para o gráfico de barras.')
else:
    ax = resumo_eventos.sort_values('registros').plot.barh(
        x='evento', y='registros', legend=False, figsize=(9, 6), color='#2563eb'
    )
    ax.set(title='Eventos com mais registros (até 15)', xlabel='Número de registros', ylabel='Evento')
    plt.tight_layout()
    plt.show()

if periodos.empty:
    print('Sem datas válidas para o gráfico mensal.')
else:
    ax = periodos.plot.line(
        x='mes', y='registros', marker='o', legend=False, figsize=(9, 4), color='#0f766e'
    )
    ax.set(title='Registros por mês', xlabel='Mês de referência', ylabel='Número de registros')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

## 11. Exercícios para a turma

1. Mostre os **10 municípios** com mais linhas, com `COUNT(*)`. Que problema pode surgir se você agrupar apenas por `municipio`?
2. Conte registros por `arma`, ignorando valores vazios, e ordene do maior para o menor.
3. Para a UF escolhida em `uf_exemplo`, conte registros por `faixa_etaria`.
4. Compare `COUNT(*)` e `COUNT(total_vitima)` por `abrangencia`. O que explica uma eventual diferença?

**Antes de abrir as soluções:** identifique as cláusulas `SELECT`, `FROM`, `WHERE`, `GROUP BY` e `ORDER BY` em cada resposta.

In [ ]:
# Solução 1 — UF + município evitam juntar homônimos de estados diferentes.
display(consultar("""
SELECT uf, municipio, COUNT(*) AS registros
FROM vde
WHERE municipio IS NOT NULL
GROUP BY uf, municipio
ORDER BY registros DESC
LIMIT 10
"""))

# Solução 2
display(consultar("""
SELECT arma, COUNT(*) AS registros
FROM vde
WHERE arma IS NOT NULL AND trim(arma) <> ''
GROUP BY arma
ORDER BY registros DESC
"""))

In [ ]:
# Solução 3 — o valor da UF entra como parâmetro.
display(consultar("""
SELECT faixa_etaria, COUNT(*) AS registros
FROM vde
WHERE uf = ?
GROUP BY faixa_etaria
ORDER BY registros DESC
""", params=(uf_exemplo,)))

# Solução 4 — COUNT(coluna) ignora NULL.
display(consultar("""
SELECT abrangencia,
       COUNT(*) AS registros,
       COUNT(total_vitima) AS registros_com_total_vitima
FROM vde
GROUP BY abrangencia
ORDER BY registros DESC
"""))

## 12. Fechamento: o que ainda precisamos validar?

- **Granularidade:** o que representa cada linha (município, evento, período, sexo, faixa etária, outro recorte)?
- **Definições:** o que significam `total_vitima`, `total`, `total_peso` e `abrangencia`? `total_peso` é valor ponderado? Não assumimos.
- **Cobertura:** existem todas as UFs e todos os meses esperados? Há linhas repetidas por recorte ou revisões?
- **Qualidade:** investigar nulos, datas inválidas e divergências entre os componentes e o total.
- **Interpretação:** comparações entre UFs por contagem bruta não controlam população, cobertura nem outros fatores.

**Fontes das ferramentas:** [Colab e Google Drive](https://colab.research.google.com/notebooks/io.ipynb), [pandas `read_excel`](https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html), [pandas `to_sql`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_sql.html), [Python `sqlite3`](https://docs.python.org/3/library/sqlite3.html).